# Frequency-based Measures

In this file, the frequency-metrics are computed for each event (baseline, fatigue-induction & control condition and drone-assembly) based on the preprocessed timeseries. 

## Load Packages & Data

In [3]:
#Load Packages
import pandas as pd
import neurokit2 as nk

In [4]:
#Load Data

## Empatica
empatica = pd.read_excel("Preprocessed_Data/Empatica_Epoch.xlsx")
empatica.head(5)


,ID,Session,Color,Station,timestamp,Local_Time,RR_Raw,Event,Group,Type,Artifact_Flag,Local_Median,RR_Deviation,RR_clean,EWMA_RR,event_start,elapsed_sec,epoch_5min,Elapsed_second_epoch,Event_Epoch
0,OITC01,2601_1,Blau,1,2026-01-26 12:59:00.443,13:59:00,515,Baseline,Control,OITC,Artifact,883.0,368.0,NaN,880.137513,2026-01-26 12:59:00.443,0.000000,1,0.000000,Baseline_1
1,OITC01,2601_1,Blau,1,2026-01-26 12:59:01.670,13:59:01,1227,Baseline,Control,OITC,Artifact,879.0,348.0,NaN,880.137513,2026-01-26 12:59:00.443,1.227258,1,1.227258,Baseline_1
2,OITC01,2601_1,Blau,1,2026-01-26 12:59:02.958,13:59:02,1288,Baseline,Control,OITC,Artifact,863.0,425.0,NaN,880.137513,2026-01-26 12:59:00.443,2.515414,1,2.515414,Baseline_1
3,OITC01,2601_1,Blau,1,2026-01-26 12:59:03.883,13:59:03,924,Baseline,Control,OITC,OK,847.0,77.0,924.0,924.000000,2026-01-26 12:59:00.443,3.439890,1,3.439890,Baseline_1
4,OITC01,2601_1,Blau,1,2026-01-26 12:59:04.724,13:59:04,842,Baseline,Control,OITC,OK,844.5,2.5,842.0,842.000000,2026-01-26 12:59:00.443,4.281581,1,4.281581,Baseline_1


In [5]:
## Polar
polar = pd.read_excel("Preprocessed_Data/Polar_Epoch.xlsx")
polar.head(5)

,ID,Session,Color,Station,Timestamp,RR_Raw,Event,Group,Type,offline,Artifact_Flag,Local_Median,RR_Deviation,RR_clean,EWMA_RR,event_start,elapsed_sec,epoch_5min,Elapsed_second_epoch,Event_Epoch
0,OITC03,2601_1,Grün,3,13:59:00.273,508,Baseline,1-Back,OITC,False,OK,509.0,1.0,508.0,508.0,50340.273,0.000,1,0.000,Baseline_1
1,OITC03,2601_1,Grün,3,13:59:00.782,509,Baseline,1-Back,OITC,False,OK,509.0,0.0,509.0,509.0,50340.273,0.509,1,0.509,Baseline_1
2,OITC03,2601_1,Grün,3,13:59:01.291,509,Baseline,1-Back,OITC,False,OK,509.0,0.0,509.0,509.0,50340.273,1.018,1,1.018,Baseline_1
3,OITC03,2601_1,Grün,3,13:59:01.802,511,Baseline,1-Back,OITC,False,OK,509.0,2.0,511.0,511.0,50340.273,1.529,1,1.529,Baseline_1
4,OITC03,2601_1,Grün,3,13:59:02.312,510,Baseline,1-Back,OITC,False,OK,509.0,1.0,510.0,510.0,50340.273,2.039,1,2.039,Baseline_1


## Compute Frequency Metrics

### For Empatica Data

#### For Split-Half-Reliability (One minute epochs)

In [10]:
# Remove missing values 
empatica = empatica.dropna(
    subset=["EWMA_RR", "Event_Epoch", "Elapsed_second_epoch"]
).copy()

# Create 1-minute intervals within each event:
empatica["Minute"] = (
    empatica["Elapsed_second_epoch"] // 60
).astype(int) + 1

results = []

# Group by participant, event, and 1-minute interval
for (participant, event_name, minute), group in empatica.groupby(
    ["ID", "Event_Epoch", "Minute"]
):

    # Sort observations chronologically
    group = group.sort_values("Elapsed_second_epoch")

    # RR intervals (milliseconds)
    rr = group["EWMA_RR"].values

    # Skip intervals with insufficient RR intervals
    if len(rr) < 10:
        continue

    try:
        # Convert RR intervals to peaks
        peaks = nk.intervals_to_peaks(
            rr,
            sampling_rate=64
        )

        # Frequency-domain HRV
        freq = nk.hrv_frequency(
            peaks,
            sampling_rate=64,
            normalize=True
        )

        # Add identifiers
        freq["ID"] = participant
        freq["Event_Epoch"] = event_name
        freq["Minute"] = minute

        results.append(freq)

    except Exception as e:
        print(
            f"Could not calculate HRV for "
            f"{participant}, {event_name}, minute {minute}: {e}"
        )


# Combine results
Frequency_HRV_1min_empatica = pd.concat(
    results,
    ignore_index=True)

# Reorder identifying columns
id_cols = ["ID", "Event_Epoch", "Minute"]
cols = id_cols + [
    c for c in Frequency_HRV_1min_empatica.columns
    if c not in id_cols]
Frequency_HRV_1min_empatica = Frequency_HRV_1min_empatica[cols]

# Remove "_1" at the end of Event_Epoch
Frequency_HRV_1min_empatica["Event_Epoch"] = Frequency_HRV_1min_empatica["Event_Epoch"].str.replace(
    r"_\d+$", "", regex=True
)

# Extract HFn
HFn_1min_empatica = Frequency_HRV_1min_empatica[
    ["ID", "Event_Epoch", "Minute", "HRV_HFn"]
].copy()

# Save only HFn
HFn_1min_empatica.to_excel(
    "Preprocessed_Data/HFn_1min_empatica.xlsx",
    index=False
)
HFn_1min_empatica

,ID,Event_Epoch,Minute,HRV_HFn
0,OITC01,Baseline,1,0.697721
1,OITC01,Baseline,2,0.764982
2,OITC01,Baseline,3,0.646558
3,OITC01,Baseline,4,0.579748
4,OITC01,Baseline,5,0.651902
...,...,...,...,...
2187,OITC50,Trial3,1,0.827956
2188,OITC50,Trial3,2,0.778267
2189,OITC50,Trial3,3,0.501300
2190,OITC50,Trial3,4,0.579654


#### For Signal & Event Level Analysis (Five minute epochs )

In [11]:
#Get HRV indicies (using Neurokit2): Empatica

## Remove missing values
empatica = empatica.dropna(subset=["EWMA_RR", "Event_Epoch"])

results = []

## Group by participant and event
for (participant, event_name), group in empatica.groupby(["ID", "Event_Epoch"]):

    # RR intervals
    rr = group["EWMA_RR"].values

    # Convert RR intervals to peaks
    peaks = nk.intervals_to_peaks(rr, sampling_rate=64)

    # Frequency-domain HRV
    freq = nk.hrv_frequency(peaks, sampling_rate=64)

    # Add identifiers
    freq["ID"] = participant
    freq["Event_Epoch"] = event_name

    results.append(freq)

## Combine all events
Empatica_Frequency_HRV = pd.concat(results, ignore_index=True)

# Reorder columns
cols = ["Event_Epoch"] + [c for c in Empatica_Frequency_HRV.columns if c != "Event_Epoch"]
Empatica_Frequency_HRV = Empatica_Frequency_HRV[cols]

# Save
Empatica_Frequency_HRV.to_excel("Preprocessed_Data/Empatica_Frequency_HRV.xlsx", index=False)

Empatica_Frequency_HRV


,Event_Epoch,HRV_ULF,HRV_VLF,HRV_LF,HRV_HF,HRV_VHF,HRV_TP,HRV_LFHF,HRV_LFn,HRV_HFn,HRV_LnHF,ID
0,Baseline_1,NaN,0.009624,0.018685,0.045172,0.008525,0.082006,0.413632,0.227845,0.550841,-3.097268,OITC01
1,Drone_1,NaN,0.016213,0.029013,0.008739,0.001746,0.055711,3.319740,0.520771,0.156871,-4.739914,OITC01
2,Drone_2,NaN,0.013256,0.017334,0.006883,0.001439,0.038912,2.518261,0.445458,0.176891,-4.978679,OITC01
3,Drone_3,NaN,0.017097,0.014811,0.011038,0.001932,0.044878,1.341899,0.330038,0.245949,-4.506450,OITC01
4,Drone_4,NaN,0.016007,0.030750,0.013337,0.002189,0.062282,2.305716,0.493724,0.214130,-4.317248,OITC01
...,...,...,...,...,...,...,...,...,...,...,...,...
442,Drone_5,NaN,0.012135,0.033058,0.023475,0.006212,0.074880,1.408193,0.441475,0.313505,-3.751811,OITC50
443,Drone_6,NaN,0.014895,0.026141,0.010526,0.002380,0.053942,2.483519,0.484623,0.195135,-4.553910,OITC50
444,Trial1_1,NaN,0.005958,0.005477,0.005877,0.000504,0.017815,0.931967,0.307428,0.329871,-5.136760,OITC50
445,Trial2_1,NaN,0.006721,0.003897,0.006504,0.000715,0.017838,0.599234,0.218488,0.364613,-5.035318,OITC50


### For Polar Data

#### For Split-Half-Reliability (One minute epochs)

In [12]:
# Remove missing values 
polar = polar.dropna(
    subset=["EWMA_RR", "Event_Epoch", "Elapsed_second_epoch"]
).copy()

# Create 1-minute intervals within each event:
polar["Minute"] = (
    polar["Elapsed_second_epoch"] // 60
).astype(int) + 1

results = []

# Group by participant, event, and 1-minute interval
for (participant, event_name, minute), group in polar.groupby(
    ["ID", "Event_Epoch", "Minute"]
):

    # Sort observations chronologically
    group = group.sort_values("Elapsed_second_epoch")

    # RR intervals (milliseconds)
    rr = group["EWMA_RR"].values

    # Skip intervals with insufficient RR intervals
    if len(rr) < 10:
        continue

    try:
        # Convert RR intervals to peaks
        peaks = nk.intervals_to_peaks(
            rr,
            sampling_rate=64
        )

        # Frequency-domain HRV
        freq = nk.hrv_frequency(
            peaks,
            sampling_rate=64,
            normalize=True
        )

        # Add identifiers
        freq["ID"] = participant
        freq["Event_Epoch"] = event_name
        freq["Minute"] = minute

        results.append(freq)

    except Exception as e:
        print(
            f"Could not calculate HRV for "
            f"{participant}, {event_name}, minute {minute}: {e}"
        )


# Combine results
Frequency_HRV_1min_polar = pd.concat(
    results,
    ignore_index=True)

# Reorder identifying columns
id_cols = ["ID", "Event_Epoch", "Minute"]
cols = id_cols + [
    c for c in Frequency_HRV_1min_polar.columns
    if c not in id_cols]
Frequency_HRV_1min_polar = Frequency_HRV_1min_polar[cols]

# Remove "_1" at the end of Event_Epoch
Frequency_HRV_1min_polar["Event_Epoch"] = Frequency_HRV_1min_polar["Event_Epoch"].str.replace(
    r"_\d+$", "", regex=True
)

# Extract HFn
HFn_1min_polar = Frequency_HRV_1min_polar[
    ["ID", "Event_Epoch", "Minute", "HRV_HFn"]
].copy()

# Save only HFn
HFn_1min_polar.to_excel(
    "Preprocessed_Data/HFn_1min_polar.xlsx",
    index=False
)
HFn_1min_polar

,ID,Event_Epoch,Minute,HRV_HFn
0,OITC03,Baseline,1,0.531844
1,OITC03,Baseline,2,0.549814
2,OITC03,Baseline,3,0.417321
3,OITC03,Baseline,4,0.391007
4,OITC03,Baseline,5,0.544742
...,...,...,...,...
1854,OITC50,Trial3,1,0.882264
1855,OITC50,Trial3,2,0.808127
1856,OITC50,Trial3,3,0.578952
1857,OITC50,Trial3,4,0.609517


#### For Signal & Event Level Analysis (Five minute epochs )

In [13]:
#Get HRV indicies (using Neurokit2): Polar

## Remove missing values
polar = polar.dropna(subset=["EWMA_RR", "Event_Epoch"])

results = []

## Group by participant and event
for (participant, event_name), group in polar.groupby(["ID", "Event_Epoch"]):

    # RR intervals
    rr = group["EWMA_RR"].values

    # Convert RR intervals to peaks
    peaks = nk.intervals_to_peaks(rr, sampling_rate=1000)

    # Frequency-domain HRV
    freq = nk.hrv_frequency(peaks, sampling_rate=1000)

    # Add identifiers
    freq["ID"] = participant
    freq["Event_Epoch"] = event_name

    results.append(freq)

## Combine all events
Polar_Frequency_HRV = pd.concat(results, ignore_index=True)

# Reorder columns
cols = ["Event_Epoch"] + [c for c in Polar_Frequency_HRV.columns if c != "Event_Epoch"]
Polar_Frequency_HRV = Polar_Frequency_HRV[cols]

# Save
Polar_Frequency_HRV.to_excel("Preprocessed_Data/Polar_Frequency_HRV.xlsx", index=False)

print(Polar_Frequency_HRV)

    Event_Epoch  HRV_ULF   HRV_VLF    HRV_LF    HRV_HF   HRV_VHF    HRV_TP  \
0    Baseline_1      NaN  0.004178  0.034578  0.038649  0.020443  0.097849   
1       Drone_1      NaN  0.007144  0.026935  0.010571  0.005723  0.050374   
2       Drone_2      NaN  0.018647  0.035989  0.008775  0.004210  0.067620   
3       Drone_3      NaN  0.009896  0.033368  0.011294  0.009948  0.064506   
4       Drone_4      NaN  0.008494  0.020836  0.008206  0.004041  0.041577   
..          ...      ...       ...       ...       ...       ...       ...   
367     Drone_5      NaN  0.009722  0.025335  0.008155  0.000794  0.044006   
368     Drone_6      NaN  0.006076  0.035532  0.006324  0.000411  0.048343   
369    Trial1_1      NaN  0.005918  0.005939  0.006790  0.000425  0.019073   
370    Trial2_1      NaN  0.007481  0.002776  0.008483  0.000556  0.019297   
371    Trial3_1      NaN  0.008497  0.006766  0.009359  0.000288  0.024912   

     HRV_LFHF   HRV_LFn   HRV_HFn  HRV_LnHF      ID  
0    0.89